# Lesson 8.4 — Capstone: The Velocity Layer for Module 7
**Module 6 · Unit 8 · Lesson 32**

Package analyzer + tracker behind one narrow interface — desired tool twists in, robust joint rates out — the velocity layer Module 7's trajectory generators will drive. Closes Module 6.

In [1]:
import numpy as np
def dh(th,d,a,al):
    ct,st,ca,sa=np.cos(th),np.sin(th),np.cos(al),np.sin(al)
    return np.array([[ct,-st*ca,st*sa,a*ct],[st,ct*ca,-ct*sa,a*st],[0,sa,ca,d],[0,0,0,1]])
def forward_chain(P,T,q):
    M=np.eye(4); Ms=[M.copy()]
    for i,(th0,d0,a,al) in enumerate(P):
        th,d=(th0+q[i],d0) if T[i]=="R" else (th0,d0+q[i]); M=M@dh(th,d,a,al); Ms.append(M.copy())
    return Ms
def geometric_jacobian(P,T,q):
    Ms=forward_chain(P,T,q); on=Ms[-1][:3,3]; J=np.zeros((6,len(q)))
    for i in range(len(q)):
        z=Ms[i][:3,2]; o=Ms[i][:3,3]
        if T[i]=="R": J[:3,i]=np.cross(z,on-o); J[3:,i]=z
        else: J[:3,i]=z
    return J
def Jv_planar(P,T,q): return geometric_jacobian(P,T,q)[:2,:]
def fk_xy(P,T,q): return forward_chain(P,T,q)[-1][:2,3]
P2=[(0,0,1,0),(0,0,1,0)]; T2=["R","R"]
P3=[(0,0,1,0),(0,0,1,0),(0,0,0.6,0)]; T3=["R","R","R"]   # redundant (2D task, 3 joints)
EPS=0.08   # singularity threshold on sigma_min


In [2]:
def analyze(P,T,q):
    """One SVD -> full capability report (Units 4-6 in one function)."""
    J=Jv_planar(P,T,q); U,S,Vt=np.linalg.svd(J)
    w=float(np.prod(S)); kappa=float(S[0]/max(S[-1],1e-12))
    return {"sigma":S,"w":w,"kappa":kappa,
            "axes":[(U[:,i],float(S[i])) for i in range(len(S))],
            "singular":bool(S[-1]<EPS),"sigma_min":float(S[-1]),"J":J,"U":U,"Vt":Vt}


In [3]:
def dls(J,lam):
    return J.T@np.linalg.inv(J@J.T+lam**2*np.eye(J.shape[0]))
def track_step(P,T,q,xi_d,dt,lam=0.0):
    """Resolve commanded tool twist -> joint rates, integrate open-loop one step."""
    J=Jv_planar(P,T,q)
    qd=(dls(J,lam) if lam>0 else np.linalg.pinv(J))@xi_d
    return q+qd*dt, qd


In [4]:
def schedule_lambda(sigma_min, lam_max=0.1, eps=EPS):
    """lambda^2 = (1-(smin/eps)^2) lam_max^2 inside the band, else 0."""
    if sigma_min>=eps: return 0.0
    return float(np.sqrt((1-(sigma_min/eps)**2))*lam_max)
def velocity_layer(P,T,q,xi_d,z=None):
    rep=analyze(P,T,q); lam=schedule_lambda(rep["sigma_min"])
    J=rep["J"]; Jd=dls(J,lam) if lam>0 else np.linalg.pinv(J)
    qd=Jd@xi_d
    if z is not None:   # null-space secondary objective (redundant arms)
        qd=qd+(np.eye(len(q))-Jd@J)@z
    info={"w":rep["w"],"kappa":rep["kappa"],"sigma_min":rep["sigma_min"],"lambda":lam,"singular":rep["singular"]}
    return qd, info


## velocity_layer(q, xi_d) -> (q_dot, info): the narrow handoff interface

In [5]:
checks=[]
q=np.array([0.4,1.2])
qd,info=velocity_layer(P2,T2,q,np.array([0.0,0.3]))
print("q_dot:",np.round(qd,3))
print("info:",{k:(round(v,3) if isinstance(v,float) else v) for k,v in info.items()})
checks.append(set(info.keys())>= {"w","kappa","sigma_min","lambda","singular"})

q_dot: [ 0.322 -0.447]
info: {'w': 0.932, 'kappa': 3.728, 'sigma_min': 0.5, 'lambda': 0.0, 'singular': False}


## Drive a pre-supplied twist stream through a near-singular stretch: bounded rates

In [6]:
q=np.array([0.4,1.4]); dt=0.01; peak=0; flagged=False
stream=[np.array([0.45,0.0]) for _ in range(300)]  # supplied by (future) Module 7
for xi_k in stream:
    qd,info=velocity_layer(P2,T2,q,xi_k); peak=max(peak,np.linalg.norm(qd))
    flagged=flagged or info["lambda"]>0; q=q+qd*dt
print(f"peak ||qdot||={peak:.2f}  damping engaged in band={flagged}")
checks.append(peak<5.0 and flagged)

peak ||qdot||=3.64  damping engaged in band=True


## Ownership of behaviors across modules (scope fence)

In [7]:
ownership={"trajectory timing (when to be where)":"Module 7",
           "pose-error correction (feedback)":"Module 8",
           "posture / null-space control":"Module 6 velocity layer (+M7 goals)"}
for k,v in ownership.items(): print(f"  {k:42s} -> {v}")
checks.append(ownership["trajectory timing (when to be where)"]=="Module 7")
print("Module 6 delivers the VELOCITY LAYER; Module 7 supplies trajectories; Module 8 adds feedback control.")
assert all(checks), f"FAILED: {checks}"
print("All checks passed.")

  trajectory timing (when to be where)       -> Module 7
  pose-error correction (feedback)           -> Module 8
  posture / null-space control               -> Module 6 velocity layer (+M7 goals)
Module 6 delivers the VELOCITY LAYER; Module 7 supplies trajectories; Module 8 adds feedback control.
All checks passed.
